## CARLA Exploration

In [2]:
import sys

In [2]:
# Need this for file control so programs can be developed outside of the installation path
f_path = r"C:\Users\mikel\Downloads\Other Important Things\RWTH-Aachen Stuff\Indpendent_Study\carla_driving_sim_work\carla\dist\carla-0.9.15-py3.7-win-amd64.egg"

agent_path = r"C:\Users\mikel\Downloads\Other Important Things\RWTH-Aachen Stuff\Indpendent_Study\carla_driving_sim_work\carla"

sys.path.append(f_path)
sys.path.append(agent_path)

In [3]:
import carla
import random
import numpy as np
import time
import cv2

In [4]:
from agents.navigation.global_route_planner import GlobalRoutePlanner

### Map Initialization

In [8]:
# Connect to the client and retrieve the world object
client = carla.Client("localhost", 2000)
world = client.get_world()

# Get the vehicle objects from the world library
vehicle_bp = world.get_blueprint_library().filter("*vehicle*")

# Get the map's spawn points
spawn_points = world.get_map().get_spawn_points()

### Spawning Vehicles/Testing Methods

In [9]:
test_vehicle_bp = world.get_blueprint_library().filter("vehicle.audi.tt")[0]
test_spawn_point = spawn_points[0]
test_vehicle = world.spawn_actor(test_vehicle_bp, test_spawn_point)

In [10]:
# Spawn 20 vehicles randomly distributed throughout the map
for i in range(20):
    vehicle = random.choice(vehicle_bp)
    location = random.choice(spawn_points)
    world.try_spawn_actor(vehicle, location)

In [11]:
# Get location/rotation before movement begins
test_vehicle_pos = test_vehicle.get_transform()
print(test_vehicle_pos)

Transform(Location(x=-64.644844, y=24.471010, z=-0.011131), Rotation(pitch=0.000000, yaw=0.159198, roll=0.000000))


In [12]:
# Access only location
print(test_vehicle_pos.location)

# Access one dimension
print(test_vehicle_pos.location.x)

Location(x=-64.644844, y=24.471010, z=-0.011131)
-64.64484405517578


In [13]:
# Define the spectator and set the location by the test vehicle
spectator = world.get_actors().filter("spectator")[0]

# If the spectator (me) is not at the test_vehicle location, update this
# Will need to revise this and generalize it, currently it only works part of the time
spectator.set_transform(carla.Transform(
    test_vehicle.get_transform().location + carla.Location(x=11, z=4),
    carla.Rotation(
        pitch=test_vehicle.get_transform().rotation.pitch - 13,
        yaw=test_vehicle.get_transform().rotation.yaw - 180
    )))

### Spawning Sensors and Testing Methods

In [18]:
# any time a modification to the block below (with spawning the camera) is made,
# another object will be spawned. Destory all cameras and then respawn one using
# block below so that only one is active at a time
for actor in world.get_actors():
    if actor.type_id == "sensor.camera.rgb":
        actor.destroy()

In [15]:
# verifying that there are no sensors currently spawned
print(world.get_actors().filter("*sensor*"))

[]


In [16]:
# previously, I was calculating the camera position globally, but it needs to be done
# relative to the vehicle so that the 'attach_to' method can work properly
relative_offset = carla.Location(x=0.5, z=1.75)

camera_transform = carla.Transform(relative_offset, test_vehicle.get_transform().rotation)

camera_bp = world.get_blueprint_library().find("sensor.camera.rgb")
# # setting image aspect ratio (not sure if this is necessary, default is 800 x 600)
# camera_bp.set_attribute("image_size_x", "1920")
# camera_bp.set_attribute("image_size_y", "1080")

camera = world.spawn_actor(camera_bp, camera_transform, attach_to=test_vehicle)

In [17]:
# since the camera object doesn't visually render, use this code to verify that 
# it is 
bound_box = carla.BoundingBox(camera.get_transform().location,
                              carla.Vector3D(0.5, 0.3, 0.3))

world.debug.draw_box(bound_box, camera.get_transform().rotation,
                     life_time=10, color=carla.Color(100,0,0))

In [126]:
# alternatively you can see this by each object's transform
for actor in world.get_actors():
    if actor.type_id == "sensor.camera.rgb":
        print(actor.get_transform())

print(test_vehicle.get_transform())

Transform(Location(x=-64.644844, y=24.471010, z=1.738811), Rotation(pitch=0.000000, yaw=0.318396, roll=0.000000))
Transform(Location(x=-64.644844, y=24.471010, z=-0.011189), Rotation(pitch=0.000000, yaw=0.159198, roll=0.000000))


Maintain speed function (from "Full Sim Driving" YouTube video)

In [128]:
print(test_vehicle)

Actor(id=138, type=vehicle.audi.tt)


In [130]:
test_vehicle.apply_control(carla.VehicleControl(throttle=0.5))

In [68]:
PREFERRED_SPEED = 20
SPEED_THRESHOLD = 2

def maintain_speed(s):
    '''
    Simple, undefined function. Open for additional refinement.
    '''
    if s >= PREFERRED_SPEED:
        return 0
    elif s < PREFERRED_SPEED - SPEED_THRESHOLD:
        return 0.8 # almost like % of "full gas"
    else:
        return 0.4

In [60]:
# Set the vehicles to the autopilot state
for vehicle in world.get_actors().filter('*vehicle*'):
    vehicle.set_autopilot(True)

In [54]:
# Get the location and rotation of a vehicle
print(test_vehicle.get_transform())

Transform(Location(x=93.410988, y=29.619640, z=-0.004365), Rotation(pitch=-0.518309, yaw=11.844669, roll=-1.972412))


In [55]:
# Get the velocity and acceleration of a vehicle
print("Vehicle velocity:", test_vehicle.get_velocity())
print("Vehicle acceleration:", test_vehicle.get_acceleration())

Vehicle velocity: Vector3D(x=0.604757, y=2.032999, z=0.000250)
Vehicle acceleration: Vector3D(x=0.272794, y=3.442598, z=-0.008956)


In [28]:
# Another example for how to get offset a location/rotation relative to an existing one
# Pay attention to the difference in style for location vs. rotation
offset_pos = carla.Transform(spectator.get_transform().location + carla.Location(z=5),
                          carla.Rotation(yaw = spectator.get_transform().rotation.yaw - 90))


In [29]:
print("Spectator position: {}".format(spectator_pos))
print("Offset position: {}".format(offset_pos))

Spectator position: Transform(Location(x=102.566200, y=43.965668, z=2.495384), Rotation(pitch=0.000061, yaw=90.390678, roll=-0.000671))
Offset position: Transform(Location(x=102.978058, y=48.993774, z=7.132627), Rotation(pitch=0.000000, yaw=-184.073303, roll=0.000000))


In [106]:
# create a route given the test vehicle location
# uses carla's built in Global Route Planner
town_map = world.get_map()
sampling_res = 2

grp = GlobalRoutePlanner(town_map, sampling_res)

start_pt = test_vehicle.get_transform().location
end_pt = test_vehicle.get_transform().location
end_pt.y += 20

route = grp.trace_route(start_pt, end_pt)

for waypoint in route:
    world.debug.draw_string(waypoint[0].transform.location, "^",
        color=carla.Color(r=0, g=0, b=255), life_time=120, 
        persistent_lines=True)

In [1]:
# Destroy all of the vehicles
for vehicle in world.get_actors().filter("*vehicle*"):
    vehicle.destroy()

NameError: name 'world' is not defined